[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter3/hallucinations.ipynb)

# Chapter 3: Hallucination Detection with HHEM

In spite of the amazing power of LLM, they still do hallucinate. In some cases, where creativity is required, hallucinations are okay or even necessary, but in most enterprise RAG use-cases a trusted response is needed.

HHEM (Hughes Hallucination Evaluation Model) is a model that was built specifically to help LLM practitioners measure hallucinations.  At a basic level, HHEM is a classification neural network model that given two text strings (sentence A and sentence B) returns a score between 0...1 reflecting the extent to which sentence B is factually consistent with sentence A.

**What you'll learn:**
- Use Vectara's HHEM model to score factual consistency
- Evaluate good and bad summaries against source text
- Interpret consistency scores for hallucination detection

**Prerequisites:** `pip install transformers`

## Setup

We import HuggingFace's pipeline and tokenizer utilities to load Vectara's HHEM hallucination detection model.

In [1]:
from transformers import pipeline, AutoTokenizer

## Define Test Cases

We set up four article-summary pairs that range from factually consistent to blatantly hallucinated, including subtle errors like wrong units and fabricated details.

In [2]:
example_pairs = [
    # Good summary
    {"article": "The woman is playing mario cart while resting on the couch",
     "summary": "The woman is playing a game resting"},

    # Bad summary: completely different summary
    {"article": "A person on a horse jumps over a broken down airplane.", 
     "summary": "A person is at a diner, ordering an omelette."},
    
    # Bad Summary: 2kg vs 2lbs, one meter vs two meters
    {"article": "Goldfish are being caught weighing up to 2kg and koi carp up to 8kg and one metre in length",
     "summary": "Koi carp can be as heavy as 2lbs and as long as two meters"}, 

    # Bad Summary: article didn't mention estimated worth
    {"article": "The plants were found during the search of a warehouse near Ashbourne on Saturday morning. Police said they were in 'an elaborate grow house'. A man in his late 40s was arrested at the scene.",
     "summary":"Police have arrested a man in his late 40s after cannabis plants worth an estimated £100,000 were found in a warehouse near Ashbourne."}
]

## Run Evaluation

We run each article-summary pair through HHEM, which returns a 0-to-1 consistency score indicating how well the summary is supported by its source text.

In [3]:
prompt = "<pad> Determine if the hypothesis is true given the premise?\n\nPremise: {text1}\n\nHypothesis: {text2}"
input_pairs = [prompt.format(text1=pair['article'], text2=pair['summary']) for pair in example_pairs]
classifier = pipeline(
            "text-classification",
            model='vectara/hallucination_evaluation_model',
            tokenizer=AutoTokenizer.from_pretrained('google/flan-t5-base'),
            trust_remote_code=True
        )
full_scores = classifier(input_pairs, top_k=None) # List[List[Dict[str, float]]]
hhem_scores = [round(score_dict['score'],4) for score_for_both_labels in full_scores for score_dict in score_for_both_labels if score_dict['label'] == 'consistent']
print(hhem_scores)

You are using a model of type HHEMv2Config to instantiate a model of type HHEMv2. This is not supported for all configurations of models and can yield errors.
You are using a model of type HHEMv2Config to instantiate a model of type HHEMv2. This is not supported for all configurations of models and can yield errors.
Device set to use mps:0


[0.9182, 0.0114, 0.1654, 0.0823]


> **Note:** HHEM scores range from 0 (hallucinated) to 1 (factually consistent). The good summary scores 0.92 — strongly consistent. The completely wrong summary scores 0.01 — clear hallucination. The subtler errors (wrong units, fabricated details) score 0.17 and 0.08, showing HHEM catches both gross and nuanced hallucinations.